### Bagging vs Boosting
We choose bagging due to limited resources, and limited data
Since boosting is prone to overfit due to sequential weighting



In [ ]:
import pandas as pd
from datetime import *
import os 
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import numpy as np


# Constants
global WORKING_DIRECTORY,RAWDATAFILES,SUPPORTED_FILEFORMATS,RAW_Data_DIR,MERGE_FILES, LABEL_DF

RAWDATAFILES={}


pd.set_option('display.max_rows', 500)

# Get the current working directory
cwd = os.getcwd()
WORKING_DIRECTORY = os.path.dirname(os.path.dirname(cwd)) #Parent directory
print("The current working directory is:", WORKING_DIRECTORY)

DATA_DIRECTORY = os.path.join(WORKING_DIRECTORY,"Project\\Data")
print("The data directory is:", DATA_DIRECTORY)

RAW_Data_DIR=  os.path.join(DATA_DIRECTORY,"01-raw")
PREPROCESS_Data_DIR=  os.path.join(DATA_DIRECTORY,"02-preprocessed")
FEATURES_Data_DIR=  os.path.join(DATA_DIRECTORY,"03-features")
PREDICTIONS_Data_DIR=  os.path.join(DATA_DIRECTORY,"04-predictions")

#import previous modules output
FEATURES_DF = pd.read_pickle(os.path.join(FEATURES_Data_DIR,"features.pkl"))
EXERCISE_DF = pd.read_pickle(os.path.join(FEATURES_Data_DIR,"exercise_dict.pkl"))

exercise_tuple = [tup for tup in EXERCISE_DF.set_index("key").itertuples()]
print(exercise_tuple)

feature_cols = FEATURES_DF.columns[:-3]
target_cols = FEATURES_DF.columns[-3:]

In [ ]:
# Class Balancing with with up and downsampling
from sklearn.utils import resample

exercise_count_tuple ={}
def class_balancing():
    global FEATURES_DF
    target_samples=92
    #Check Amount of Sample per Class
    for idx, c in FEATURES_DF.groupby("target").agg({"target":'count'}).rename(columns={'target': 'target_count'}).iterrows():
        exercise_count_tuple[idx] = c["target_count"]

    for element in exercise_count_tuple:
        df_keep = FEATURES_DF[FEATURES_DF.target!=element].copy()
        df_adjust = FEATURES_DF[FEATURES_DF.target==element].copy()
        df_upsampled=pd.DataFrame()
        
        if exercise_count_tuple[element] < target_samples:
            #upsample
            
            #avoid overfitting:
            if False:
                df_upsampled = resample(df_adjust,
                                        replace=True,
                                        n_samples=target_samples,    
                                        random_state=42) 
                print(df_upsampled.shape)
                Feature_DF = pd.concat([df_keep, df_upsampled])
        else:
            #downsample            
            df_downsampled=resample(df_adjust,    replace=False, 
                                    n_samples=target_samples, 
                                    random_state=42) 
            print(df_upsampled.shape)
            FEATURES_DF= pd.concat([df_keep, df_downsampled])

class_balancing()


In [ ]:

ClassFilter=[2, 3,  12, 22,25,35]
mask = (FEATURES_DF.target.isin( ClassFilter))

for col in feature_cols:
    FEATURES_DF[col] = FEATURES_DF[col].astype(float)

FEATURES_DF["target"]= pd.Series(FEATURES_DF["target"], dtype=pd.Int64Dtype())
FEATURES_DF.info()

X_train, X_test, y_train, y_test = train_test_split(FEATURES_DF[mask][feature_cols], FEATURES_DF[mask].target, stratify=FEATURES_DF[mask].target, random_state=42,shuffle=True) # stratify preserves the class ratios

clfSVC = BaggingClassifier(estimator=SVC(kernel='poly', C=1, gamma="auto"),n_estimators=30, random_state=12,n_jobs=-1).fit(X_train, y_train)
y_SVC=clfSVC.predict(X_test)
print("SVC: " ,clfSVC.score(X_test,y_test))

clfDTC = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=32),n_estimators=30, random_state=12,n_jobs=-1).fit(X_train, y_train)
y_DTC = clfDTC.predict(X_test)
print("Decision Tree: " ,clfDTC.score(X_test,y_test))

clfRFC = BaggingClassifier(estimator=RandomForestClassifier(),n_estimators=40, random_state=12,n_jobs=-1).fit(X_train, y_train)
y_RFC=clfRFC.predict(X_test)
print("RandomForest: ", clfRFC.score(X_test,y_test))



In [ ]:
print(clfSVC.classes_)
print(EXERCISE_DF[EXERCISE_DF.index.isin(ClassFilter)]["value"])

In [ ]:
def plotCM(y_Truth,y_Predicted,labels,clf_classes,title):
    cm = confusion_matrix(y_Truth, y_Predicted,labels=clf_classes) 
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                display_labels=labels)
    disp.plot()
    plt.xticks(rotation=30)
    plt.title(title)

plotCM(y_test, y_SVC,EXERCISE_DF[EXERCISE_DF.index.isin(clfSVC.classes_)]["value"],clfSVC.classes_,"Support Vector Classifier")
plotCM(y_test, y_DTC,EXERCISE_DF[EXERCISE_DF.index.isin(clfDTC.classes_)]["value"],clfDTC.classes_,"Decision Tree Classifier")
plotCM(y_test, y_RFC,EXERCISE_DF[EXERCISE_DF.index.isin(clfRFC.classes_)]["value"],clfRFC.classes_,"Random Forest Classifier")
plt.show()

In [ ]:
print("classes used: ", list(EXERCISE_DF[EXERCISE_DF.target_count>40].index))


In [ ]:


from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

#filter classes  with few samples (<40)
mask =((FEATURES_DF.target>1) & (FEATURES_DF.target.isin(list(EXERCISE_DF[EXERCISE_DF.target_count>40].index))))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(FEATURES_DF[mask][feature_cols])
pca = PCA(.9)
X_pca = pca.fit_transform(X_scaled)

X_train, X_test, y_train, y_test = train_test_split(X_pca, FEATURES_DF[mask].target, stratify=FEATURES_DF[mask].target, test_size=0.2, random_state=42,shuffle=True) # stratify preserves the class ratios

clfSVC = BaggingClassifier(estimator=SVC(kernel='poly', C=1, gamma="auto"),n_estimators=80, random_state=42,n_jobs=-1,oob_score=True).fit(X_train, y_train)
y_SVC=clfSVC.predict(X_test)
print("SVC: " ,clfSVC.score(X_test,y_test))

clfDTC = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=24),n_estimators=80, random_state=42,n_jobs=-1,oob_score=True).fit(X_train, y_train)
y_DTC = clfDTC.predict(X_test)
print("Decision Tree: " ,clfDTC.score(X_test,y_test))

clfRFC = BaggingClassifier(estimator=RandomForestClassifier(),n_estimators=80, random_state=42,n_jobs=-1,oob_score=True).fit(X_train, y_train)
y_RFC=clfRFC.predict(X_test)
print("RandomForest: ", clfRFC.score(X_test,y_test))

# without bootstrap and oob 2.5 80 N-estimators
#SVC:  0.92018779342723
# Decision Tree:  0.8849765258215962
# RandomForest:  0.892018779342723

# Without bootstrap and oob 5 80 estimators
# SVC:  0.863849765258216
# Decision Tree:  0.8028169014084507
# RandomForest:  0.8450704225352113


In [ ]:

def print_report(y_test, y_SVC):
    report = classification_report(y_test, y_SVC, output_dict=True)
    df = pd.DataFrame(report).transpose()
    print(df)
print_report(y_test, y_SVC)
print_report(y_test, y_DTC)
print_report(y_test, y_RFC)

In [ ]:
def plotCM(y_Truth,y_Predicted,labels,clf_classes,title):
    plt.figure(figsize=(20, 15))
    cm = confusion_matrix(y_Truth, y_Predicted,labels=clf_classes) 
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                display_labels=labels)
    disp.plot()

    plt.xticks(rotation=90)
    plt.title(title)
plotCM(y_test, y_SVC,EXERCISE_DF[EXERCISE_DF.index.isin(clfSVC.classes_)]["value"],clfSVC.classes_,"Support Vector Classifier")
plotCM(y_test, y_DTC,EXERCISE_DF[EXERCISE_DF.index.isin(clfDTC.classes_)]["value"],clfDTC.classes_,"Decision Tree Classifier")
plotCM(y_test, y_RFC,EXERCISE_DF[EXERCISE_DF.index.isin(clfRFC.classes_)]["value"],clfRFC.classes_,"Random Forest Classifier")
plt.show()




In [ ]:

clfRFC.oob_score
